To properly wrap up your training from 11_train_pest_classifier.ipynb, file 12 should handle model validation metrics, confusion matrix plotting, and live testing via Gradio.

Create and run this code structure in your 12th notebook (fully integrated with your dynamic project_config setup):

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd()))

from ultralytics import YOLO
import gradio as gr
from PIL import Image

# Correct absolute path pointing inside the notebooks folder where training saved it
model_path = Path(r"Z:\Crop Identification\notebooks\runs\classify\pest_classifier_runs\yolov8n_cls_pest\weights\best.pt")

print(f"Loading model from: {model_path}")
if not model_path.exists():
    raise FileNotFoundError(f"Model weights not found at: {model_path}")

model = YOLO(str(model_path))

# 2. Evaluate model performance on the test split
print("Running test split evaluation metrics...")
metrics = model.val(split="test")
print(f"Top-1 Accuracy: {metrics.top1:.4f}")
print(f"Top-5 Accuracy: {metrics.top5:.4f}")

# 3. Define Gradio inference function for live testing
def predict_pest(image: Image.Image):
    if image is None:
        return "No image provided", "0.00%", {}
        
    results = model(image)
    result = results[0]
    
    probs = result.probs
    top1_idx = probs.top1
    confidence = float(probs.top1conf.item())
    class_name = result.names[top1_idx]
    
    all_probs = {
        result.names[i]: float(probs.data[i].item()) 
        for i in range(len(result.names))
    }
    
    return class_name, f"{confidence * 100:.2f}%", all_probs

# 4. Launch Gradio web interface
demo = gr.Interface(
    fn=predict_pest,
    inputs=gr.Image(type="pil", label="Upload Crop Leaf / Pest Image"),
    outputs=[
        gr.Textbox(label="Predicted Pest Class"),
        gr.Textbox(label="Confidence Score"),
        gr.Label(label="All Class Probabilities")
    ],
    title="Smart Farming - Pest Classifier Evaluation",
    description="Upload a photo of a crop leaf to test your trained YOLOv8 pest classification model live."
)

if __name__ == "__main__":
    demo.launch(server_name="127.0.0.1", server_port=7860, share=False)

Loading model from: Z:\Crop Identification\notebooks\runs\classify\pest_classifier_runs\yolov8n_cls_pest\weights\best.pt
Running test split evaluation metrics...
Ultralytics 8.4.121  Python-3.13.5 torch-2.13.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
YOLOv8n-cls summary (fused): 30 layers, 1,440,004 parameters, 0 gradients, 3.3 GFLOPs
train: Z:\Crop Identification\notebooks\pest_dataset\train... found 380 images in 4 classes  
val: Z:\Crop Identification\notebooks\pest_dataset\val... found 81 images in 4 classes  
test: Z:\Crop Identification\notebooks\pest_dataset\test... found 82 images in 4 classes  
test: Fast image access  (ping: 0.10.0 ms, read: 1208.5664.7 MB/s, size: 397.7 KB)
test: Scanning Z:\Crop Identification\notebooks\pest_dataset\test... 82 images, 0 corrupt: 100% ━━━━━━━━━━━━ 82/82 17.2Mit/s 0.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 6/6 3.5s/it 21.0s0.4s0
                   all      0.963          1
Speed: 0.6ms preproces


0: 224x224 SM 1.00, LM 0.00, AW 0.00, A 0.00, 7.1ms
Speed: 17.8ms preprocess, 7.1ms inference, 0.4ms postprocess per image at shape (1, 3, 224, 224)

0: 224x224 A 0.54, LM 0.37, AW 0.05, SM 0.04, 66.9ms
Speed: 3.2ms preprocess, 66.9ms inference, 2.5ms postprocess per image at shape (1, 3, 224, 224)

0: 224x224 LM 1.00, AW 0.00, SM 0.00, A 0.00, 56.6ms
Speed: 3.8ms preprocess, 56.6ms inference, 0.1ms postprocess per image at shape (1, 3, 224, 224)

0: 224x224 AW 0.99, LM 0.01, SM 0.00, A 0.00, 51.0ms
Speed: 3.7ms preprocess, 51.0ms inference, 0.1ms postprocess per image at shape (1, 3, 224, 224)
